# Fine-tune DeBERTa-v3-base chấm điểm CEFR từ transcript

**Người thực hiện:** Lê Văn Chiến Thắng  
**Dataset:** Speak & Improve Corpus 2025  
**Bài toán:** hồi quy điểm SLA 0–6 cho P1, P3, P4, P5  
**Model:** `microsoft/deberta-v3-base`

Notebook này được thiết kế để chạy trên **Google Colab T4**. Mỗi phần được lưu riêng lên Google Drive. Nếu Colab ngắt, chạy lại từ đầu; phần đã có `metrics.json` và `dev_predictions.tsv` sẽ được bỏ qua.

> Quyền dữ liệu: không chia sẻ corpus, transcript, audio hoặc model weights lên GitHub/Hugging Face. Notebook chỉ in thống kê tổng hợp, không in transcript.

## 0. Trước khi chạy

1. Upload cả thư mục `sandi-corpus-2025` vào `MyDrive/sandi-corpus-2025`.
2. Trong Colab chọn **Runtime → Change runtime type → T4 GPU**.
3. Chạy lần lượt từ trên xuống bằng **Runtime → Run all**.
4. Lần đầu Google Drive sẽ hỏi quyền truy cập; chọn tài khoản sở hữu corpus.

Kết quả và checkpoint được lưu tại `MyDrive/aicefr-thang/deberta-results`, không nằm trong Git.

In [1]:
# Cài đúng bộ phiên bản đã kiểm tra khi tạo notebook. Không cài lại torch:
# Colab cung cấp torch tương thích với GPU của runtime.
%pip install -q "transformers==5.17.0" "datasets==5.0.1" "accelerate==1.15.0"
%pip install -q "sentencepiece==0.2.2" "scipy>=1.11" "pandas>=2.2"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 47.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 5.3 MB/s eta 0:00:00


In [2]:
import gc
import json
import math
import os
import platform
from collections import Counter, defaultdict
from pathlib import Path

import accelerate
import datasets
import numpy as np
import pandas as pd
import torch
import transformers
from scipy.stats import pearsonr
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    set_seed,
)
from transformers.trainer_utils import get_last_checkpoint

os.environ["TOKENIZERS_PARALLELISM"] = "false"

print("Python       :", platform.python_version())
print("PyTorch      :", torch.__version__)
print("Transformers :", transformers.__version__)
print("Datasets     :", datasets.__version__)
print("Accelerate   :", accelerate.__version__)
print("CUDA         :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU          :", torch.cuda.get_device_name(0))
    print("VRAM (GB)    :", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1))
else:
    raise RuntimeError("Chưa bật GPU. Vào Runtime → Change runtime type → T4 GPU rồi chạy lại.")

Python       : 3.13.15
PyTorch      : 2.11.0+cu128
Transformers : 5.17.0
Datasets     : 5.0.1
Accelerate   : 1.15.0
CUDA         : True
GPU          : Tesla T4
VRAM (GB)    : 14.6


## 1. Gắn Drive và cấu hình

`DATA_ROOT` phải trỏ tới thư mục đã giải nén, nơi có `reference-materials/pre-norm` và `reference-materials/sla-marks`.

In [3]:
from google.colab import drive

drive.mount("/content/drive")

DATA_ROOT = Path("/content/drive/MyDrive/sandi-corpus-2025")
OUTPUT_ROOT = Path("/content/drive/MyDrive/aicefr-thang/deberta-results")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

MODEL_NAME = "microsoft/deberta-v3-base"
PARTS = [1, 3, 4, 5]
SEED = 42
NUM_EPOCHS = 3
LEARNING_RATE = 2e-5
TRAIN_BATCH_SIZE = 8
GRADIENT_ACCUMULATION = 2  # batch hiệu dụng = 16
EVAL_BATCH_SIZE = 16
MAX_LENGTH = 512
SKIP_COMPLETED = True
SAVE_BEST_MODEL = True

required_dirs = [
    DATA_ROOT / "reference-materials/pre-norm",
    DATA_ROOT / "reference-materials/sla-marks",
]
missing_dirs = [str(path) for path in required_dirs if not path.is_dir()]
if missing_dirs:
    raise FileNotFoundError(
        "Không tìm thấy corpus đã giải nén. Thiếu:\n- " + "\n- ".join(missing_dirs)
    )

print("Corpus :", DATA_ROOT)
print("Output :", OUTPUT_ROOT)
print("Parts  :", PARTS)

Mounted at /content/drive
Corpus : /content/drive/MyDrive/sandi-corpus-2025
Output : /content/drive/MyDrive/aicefr-thang/deberta-results
Parts  : [1, 3, 4, 5]


## 2. Đọc CTM và ghép nhãn

ID bài nộp là 12 ký tự đầu của cột 1 trong CTM, đúng theo script baseline của corpus. Cell kiểm tra chặt các lỗi thường gặp: dòng CTM hỏng, ID trùng, transcript thiếu nhãn, nhãn thiếu transcript, điểm ngoài 0–6 và rò ID giữa train/dev.

In [4]:
def ctm_path(split: str, part: int) -> Path:
    return DATA_ROOT / f"reference-materials/pre-norm/{split}-sla-P{part}.ctm"


def marks_path(split: str, part: int | str) -> Path:
    part_name = f"P{part}" if isinstance(part, int) else part
    return DATA_ROOT / f"reference-materials/sla-marks/{split}-sla-{part_name}.tsv"


def read_ctm(split: str, part: int) -> tuple[dict[str, str], dict]:
    path = ctm_path(split, part)
    if not path.is_file():
        raise FileNotFoundError(path)

    words_by_id = defaultdict(list)
    malformed = 0
    part_markers = Counter()

    with path.open(encoding="utf-8") as handle:
        for line in handle:
            columns = line.split()
            if len(columns) not in (5, 6):
                malformed += 1
                continue

            segment_id = columns[0]
            submission_id = segment_id[:12]
            word = columns[4]
            if len(segment_id) > 14:
                part_markers[segment_id[14]] += 1
            words_by_id[submission_id].append(word)

    transcripts = {key: " ".join(words) for key, words in words_by_id.items()}
    metadata = {
        "path": str(path),
        "malformed_lines": malformed,
        "part_markers": dict(part_markers),
    }
    return transcripts, metadata


def read_marks(split: str, part: int | str) -> dict[str, float]:
    path = marks_path(split, part)
    if not path.is_file():
        raise FileNotFoundError(path)

    labels = {}
    with path.open(encoding="utf-8") as handle:
        for line_number, line in enumerate(handle, start=1):
            if not line.strip():
                continue
            columns = line.rstrip("\n").split("\t")
            if len(columns) < 2:
                raise ValueError(f"Dòng nhãn lỗi tại {path}:{line_number}")
            submission_id, score_text = columns[0], columns[1]
            if submission_id in labels:
                raise ValueError(f"ID nhãn bị trùng: {submission_id}")
            score = float(score_text)
            if not 0.0 <= score <= 6.0:
                raise ValueError(f"Điểm ngoài thang 0–6 tại {path}:{line_number}")
            labels[submission_id] = score
    return labels


def load_part(split: str, part: int) -> pd.DataFrame:
    transcripts, ctm_meta = read_ctm(split, part)
    labels = read_marks(split, part)

    text_ids = set(transcripts)
    label_ids = set(labels)
    text_without_label = sorted(text_ids - label_ids)
    label_without_text = sorted(label_ids - text_ids)
    if text_without_label or label_without_text:
        raise ValueError(
            f"{split} P{part}: không ghép kín ID; "
            f"text_without_label={len(text_without_label)}, "
            f"label_without_text={len(label_without_text)}"
        )

    ids = sorted(text_ids & label_ids)
    frame = pd.DataFrame({
        "submission_id": ids,
        "text": [transcripts[key] for key in ids],
        "label": [labels[key] for key in ids],
    })
    frame.attrs["ctm_meta"] = ctm_meta
    return frame

In [5]:
# Kiểm toán dữ liệu bằng số liệu tổng hợp; không hiển thị transcript.
data_cache = {}
audit_rows = []

for part in PARTS:
    train_df = load_part("train", part)
    dev_df = load_part("dev", part)
    overlap = set(train_df.submission_id) & set(dev_df.submission_id)
    if overlap:
        raise ValueError(f"P{part}: có {len(overlap)} ID xuất hiện ở cả train và dev")

    expected_marker = str(part)
    for split_name, frame in (("train", train_df), ("dev", dev_df)):
        markers = frame.attrs["ctm_meta"]["part_markers"]
        if set(markers) != {expected_marker}:
            raise ValueError(f"{split_name} P{part}: marker phần thi bất thường: {markers}")
        if frame.attrs["ctm_meta"]["malformed_lines"]:
            raise ValueError(f"{split_name} P{part}: CTM có dòng sai định dạng")

        audit_rows.append({
            "split": split_name,
            "part": f"P{part}",
            "samples": len(frame),
            "score_min": frame.label.min(),
            "score_max": frame.label.max(),
            "score_mean": frame.label.mean(),
            "words_median": frame.text.str.split().str.len().median(),
            "train_dev_overlap": len(overlap),
        })

    data_cache[part] = {"train": train_df, "dev": dev_df}

audit_df = pd.DataFrame(audit_rows)
display(audit_df.round(3))

expected_counts = {
    ("train", 1): 3068, ("train", 3): 3060,
    ("train", 4): 3005, ("train", 5): 3085,
    ("dev", 1): 438, ("dev", 3): 438,
    ("dev", 4): 438, ("dev", 5): 438,
}
for part in PARTS:
    for split_name in ("train", "dev"):
        actual = len(data_cache[part][split_name])
        expected = expected_counts[(split_name, part)]
        assert actual == expected, f"{split_name} P{part}: cần {expected}, nhận {actual}"

print("✓ Corpus khớp gói đã kiểm tra: đủ 12.218 train và 1.752 dev theo bốn phần.")

,split,part,samples,score_min,score_max,score_mean,words_median,train_dev_overlap
0,train,P1,3068,2.0,5.5,3.953,147.0,0
1,dev,P1,438,2.0,5.5,3.844,148.0,0
2,train,P3,3060,2.0,5.5,4.136,94.5,0
3,dev,P3,438,2.0,5.5,4.031,92.0,0
4,train,P4,3005,2.0,5.5,3.861,87.0,0
5,dev,P4,438,2.0,5.5,3.863,87.0,0
6,train,P5,3085,2.0,5.5,3.939,165.0,0
7,dev,P5,438,2.0,5.5,3.888,160.0,0


✓ Corpus khớp gói đã kiểm tra: đủ 12.218 train và 1.752 dev theo bốn phần.


## 3. Dataset, tokenizer và metric

Dùng dynamic padding để giảm VRAM. Batch vật lý 8 và tích lũy gradient 2 lần cho batch hiệu dụng 16. Metric chính là Pearson correlation (PCC); RMSE được báo cùng để đo sai số tuyệt đối.

In [6]:
set_seed(SEED)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer, pad_to_multiple_of=8)


class CefrRegressionDataset(torch.utils.data.Dataset):
    def __init__(self, frame: pd.DataFrame):
        self.ids = frame.submission_id.tolist()
        self.labels = frame.label.astype(float).tolist()
        self.encodings = tokenizer(
            frame.text.tolist(),
            truncation=True,
            max_length=MAX_LENGTH,
            padding=False,
        )

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):
        item = {key: values[index] for key, values in self.encodings.items()}
        item["labels"] = float(self.labels[index])
        return item


def regression_metrics(predictions, references) -> dict[str, float]:
    predictions = np.asarray(predictions, dtype=float).reshape(-1)
    references = np.asarray(references, dtype=float).reshape(-1)
    return {
        "pcc": float(pearsonr(references, predictions).statistic),
        "rmse": float(np.sqrt(np.mean((predictions - references) ** 2))),
    }


def compute_metrics(eval_prediction) -> dict[str, float]:
    return regression_metrics(
        eval_prediction.predictions.squeeze(),
        eval_prediction.label_ids,
    )

print("Tokenizer sẵn sàng:", MODEL_NAME)

config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model: reconstructing file:   0%|          |  0.00B / 2.46MB            

spm.model: downloading bytes:           |  0.00B            

Tokenizer sẵn sàng: microsoft/deberta-v3-base


## 4. Hàm huấn luyện có checkpoint và resume

Mỗi P dùng một model mới hoàn toàn. Không huấn luyện tiếp P3 từ model P1. Checkpoint tốt nhất được chọn theo PCC trên dev.

In [7]:
def run_part(part: int) -> tuple[dict, pd.DataFrame]:
    part_dir = OUTPUT_ROOT / f"P{part}"
    checkpoint_dir = part_dir / "checkpoints"
    metrics_path = part_dir / "metrics.json"
    predictions_path = part_dir / "dev_predictions.tsv"
    part_dir.mkdir(parents=True, exist_ok=True)

    if SKIP_COMPLETED and metrics_path.is_file() and predictions_path.is_file():
        print(f"P{part}: đã hoàn thành, đọc kết quả từ Drive.")
        metrics = json.loads(metrics_path.read_text(encoding="utf-8"))
        predictions = pd.read_csv(predictions_path, sep="\t")
        return metrics, predictions

    train_df = data_cache[part]["train"]
    dev_df = data_cache[part]["dev"]
    train_dataset = CefrRegressionDataset(train_df)
    dev_dataset = CefrRegressionDataset(dev_df)

    set_seed(SEED + part)
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=1,
        problem_type="regression",
        dtype=torch.float32,
    )
    # Giữ master weights/gradients ở FP32; Trainer sẽ tự autocast phép tính sang FP16.
    # Nếu để Transformers 5.x tự chọn dtype từ checkpoint, GradScaler có thể nhận
    # gradient FP16 và dừng với lỗi 'Attempting to unscale FP16 gradients'.
    model.float()
    trainable_dtypes = {param.dtype for param in model.parameters() if param.requires_grad}
    if trainable_dtypes != {torch.float32}:
        raise RuntimeError(f"Trọng số trainable phải là FP32, đang nhận: {trainable_dtypes}")
    model.gradient_checkpointing_enable()

    updates_per_epoch = math.ceil(
        len(train_dataset) / (TRAIN_BATCH_SIZE * GRADIENT_ACCUMULATION)
    )
    warmup_steps = max(1, math.ceil(updates_per_epoch * NUM_EPOCHS * 0.10))

    training_args = TrainingArguments(
        output_dir=str(checkpoint_dir),
        num_train_epochs=NUM_EPOCHS,
        learning_rate=LEARNING_RATE,
        weight_decay=0.01,
        warmup_steps=warmup_steps,
        per_device_train_batch_size=TRAIN_BATCH_SIZE,
        gradient_accumulation_steps=GRADIENT_ACCUMULATION,
        per_device_eval_batch_size=EVAL_BATCH_SIZE,
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=1,
        load_best_model_at_end=True,
        metric_for_best_model="pcc",
        greater_is_better=True,
        logging_strategy="steps",
        logging_steps=50,
        fp16=True,
        bf16=False,
        gradient_checkpointing=True,
        report_to=[],
        seed=SEED + part,
        data_seed=SEED + part,
        run_name=f"deberta-cefr-P{part}",
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=dev_dataset,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
        processing_class=tokenizer,
    )

    last_checkpoint = get_last_checkpoint(str(checkpoint_dir)) if checkpoint_dir.is_dir() else None
    if last_checkpoint:
        print(f"P{part}: tiếp tục từ {last_checkpoint}")
    else:
        print(f"P{part}: bắt đầu model mới với {len(train_df)} train / {len(dev_df)} dev")

    trainer.train(resume_from_checkpoint=last_checkpoint)
    prediction_output = trainer.predict(dev_dataset)
    raw_predictions = prediction_output.predictions.squeeze().astype(float)
    clipped_predictions = np.clip(raw_predictions, 0.0, 6.0)
    metrics = regression_metrics(clipped_predictions, dev_df.label.to_numpy())
    metrics.update({
        "part": f"P{part}",
        "train_samples": len(train_df),
        "dev_samples": len(dev_df),
        "model_name": MODEL_NAME,
        "seed": SEED + part,
        "epochs": NUM_EPOCHS,
        "learning_rate": LEARNING_RATE,
        "transformers_version": transformers.__version__,
        "torch_version": torch.__version__,
        "best_checkpoint": trainer.state.best_model_checkpoint,
    })

    predictions = pd.DataFrame({
        "submission_id": dev_df.submission_id,
        "reference": dev_df.label,
        "prediction_raw": raw_predictions,
        "prediction_clipped": clipped_predictions,
    })
    predictions.to_csv(predictions_path, sep="\t", index=False)
    metrics_path.write_text(json.dumps(metrics, indent=2, ensure_ascii=False), encoding="utf-8")

    if SAVE_BEST_MODEL:
        best_model_dir = part_dir / "best-model"
        trainer.save_model(str(best_model_dir))
        tokenizer.save_pretrained(str(best_model_dir))

    print(f"P{part}: PCC={metrics['pcc']:.4f} · RMSE={metrics['rmse']:.4f}")

    del trainer, model, train_dataset, dev_dataset
    gc.collect()
    torch.cuda.empty_cache()
    return metrics, predictions

## 5. Huấn luyện P1, P3, P4, P5

Cell này có thể mất khoảng 2–3 giờ cho cả bốn phần. Kết quả từng phần được lưu ngay sau khi phần đó hoàn tất. Nếu muốn smoke test trước, tạm đổi `PARTS_TO_RUN = [1]`; sau khi P1 chạy ổn, đổi lại `[1, 3, 4, 5]`.

In [8]:
PARTS_TO_RUN = [1, 3, 4, 5]

part_metrics = {}
part_predictions = {}

for part in PARTS_TO_RUN:
    metrics, predictions = run_part(part)
    part_metrics[part] = metrics
    part_predictions[part] = predictions

part_summary = pd.DataFrame([part_metrics[part] for part in PARTS_TO_RUN])
display(part_summary[["part", "train_samples", "dev_samples", "pcc", "rmse"]].round(4))

P1: đã hoàn thành, đọc kết quả từ Drive.
P3: đã hoàn thành, đọc kết quả từ Drive.
P4: đã hoàn thành, đọc kết quả từ Drive.
P5: đã hoàn thành, đọc kết quả từ Drive.


,part,train_samples,dev_samples,pcc,rmse
0,P1,3068,438,0.7847,0.6167
1,P3,3060,438,0.7238,0.5420
2,P4,3005,438,0.6906,0.6756
3,P5,3085,438,0.7999,0.5964


## 6. Ghép bốn phần và tính điểm overall

Điểm dự đoán overall là trung bình dự đoán P1/P3/P4/P5 theo cùng `submission_id`, rồi so với `dev-sla-overall.tsv`. Cell sẽ từ chối chạy nếu chưa có đủ bốn phần hoặc ID không khớp.

In [9]:
if set(part_predictions) != set(PARTS):
    raise RuntimeError("Cần chạy đủ P1, P3, P4, P5 trước khi tính overall.")

merged = None
for part in PARTS:
    current = part_predictions[part][["submission_id", "prediction_clipped"]].rename(
        columns={"prediction_clipped": f"prediction_P{part}"}
    )
    merged = (
        current
        if merged is None
        else merged.merge(
            current, on="submission_id", how="inner", validate="one_to_one"
        )
    )

overall_labels = read_marks("dev", "overall")
merged["reference_overall"] = merged.submission_id.map(overall_labels)
if merged.reference_overall.isna().any():
    raise ValueError("Có submission dev không tìm thấy nhãn overall.")
if len(merged) != len(overall_labels):
    raise ValueError(f"Overall cần {len(overall_labels)} ID, chỉ ghép được {len(merged)}")

prediction_columns = [f"prediction_P{part}" for part in PARTS]
merged["prediction_overall"] = merged[prediction_columns].mean(axis=1)
overall_metrics = regression_metrics(merged.prediction_overall, merged.reference_overall)
overall_metrics.update({
    "part": "overall",
    "dev_samples": len(merged),
    "aggregation": "mean(P1,P3,P4,P5)",
    "target_pcc": 0.740,
    "beats_bert_baseline": bool(overall_metrics["pcc"] > 0.740),
})

merged.to_csv(OUTPUT_ROOT / "dev_overall_predictions.tsv", sep="\t", index=False)
(OUTPUT_ROOT / "overall_metrics.json").write_text(
    json.dumps(overall_metrics, indent=2, ensure_ascii=False), encoding="utf-8"
)

print(f"OVERALL: PCC={overall_metrics['pcc']:.4f} · RMSE={overall_metrics['rmse']:.4f}")
print("Vượt baseline BERT PCC 0.740:", "CÓ" if overall_metrics["beats_bert_baseline"] else "CHƯA")

OVERALL: PCC=0.8273 · RMSE=0.5118
Vượt baseline BERT PCC 0.740: CÓ


## 7. Xuất báo cáo tái lập

Chỉ xuất cấu hình, thống kê và metric. Không xuất transcript vào notebook/report.

In [10]:
summary_rows = [
    {"part": f"P{part}", "pcc": part_metrics[part]["pcc"], "rmse": part_metrics[part]["rmse"]}
    for part in PARTS
]
summary_rows.append(
    {"part": "overall", "pcc": overall_metrics["pcc"], "rmse": overall_metrics["rmse"]}
)
summary_df = pd.DataFrame(summary_rows)
summary_df["beats_0.740"] = summary_df.pcc > 0.740
summary_df.to_csv(OUTPUT_ROOT / "metrics_summary.csv", index=False)

manifest = {
    "owner": "Le Van Chien Thang",
    "task": "CEFR transcript score regression",
    "model_name": MODEL_NAME,
    "parts": PARTS,
    "seed": SEED,
    "epochs": NUM_EPOCHS,
    "learning_rate": LEARNING_RATE,
    "physical_train_batch": TRAIN_BATCH_SIZE,
    "gradient_accumulation": GRADIENT_ACCUMULATION,
    "effective_train_batch": TRAIN_BATCH_SIZE * GRADIENT_ACCUMULATION,
    "max_length": MAX_LENGTH,
    "python_version": platform.python_version(),
    "torch_version": torch.__version__,
    "transformers_version": transformers.__version__,
    "datasets_version": datasets.__version__,
    "accelerate_version": accelerate.__version__,
    "gpu": torch.cuda.get_device_name(0),
    "dataset_counts": audit_df.to_dict(orient="records"),
    "metrics": summary_rows,
}
(OUTPUT_ROOT / "run_manifest.json").write_text(
    json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8"
)

display(summary_df.round(4))
print("✓ Đã lưu báo cáo tại:", OUTPUT_ROOT)

,part,pcc,rmse,beats_0.740
0,P1,0.7847,0.6167,True
1,P3,0.7238,0.5420,False
2,P4,0.6906,0.6756,False
3,P5,0.7999,0.5964,True
4,overall,0.8273,0.5118,True


✓ Đã lưu báo cáo tại: /content/drive/MyDrive/aicefr-thang/deberta-results


## 8. Bàn giao

Sau khi chạy xong:

1. Kiểm tra cell cuối có đủ P1, P3, P4, P5 và overall.
2. Trong Colab chọn **File → Download → Download .ipynb** để giữ notebook có output metric.
3. Có thể commit notebook và báo cáo metric tổng hợp.
4. **Không commit** thư mục corpus, `dev_predictions.tsv`, checkpoint hoặc `best-model`. Các file này đang ở Drive và phải giữ ngoài Git theo giấy phép.
5. Báo nhóm: GPU, phiên bản thư viện, số mẫu, PCC/RMSE từng phần và overall, seed, số epoch.

Mốc so sánh của nhóm: BERT PCC 0.740. Kết quả thấp hơn vẫn phải báo trung thực và phân tích; không chọn lại seed chỉ để lấy số đẹp.